# 03. panel

## 0. setup

In [1]:
import gc
from pathlib import Path
 
import numpy as np
import pandas as pd

# paths
root         = Path.cwd().parent
data_raw     = root / 'data' / 'raw'
data_interim = root / 'data' / 'interim'
data_proc    = root / 'data' / 'processed'

data_proc.mkdir(parents=True, exist_ok=True)

## 1. config

In [2]:
year_max = 2023        # hard cut: 2024 is publication-lag corrupted

in_treat  = data_proc    / 'cartel_treated.parquet'   # infr x nace3 x ctry
in_pat    = data_interim / 'pat_panel.parquet'        # ctry x isic3 x year
out_panel = data_proc    / 'panel.parquet'

# EEA-31 (ever-member) universe; countries outside = geographic-Europe controls (in_eea=0)
EEA31 = frozenset({'AT','BE','DK','FI','FR','DE','GR','IS','IE','IT','LU','NL','NO',
                   'PT','ES','SE','GB','LI','CY','CZ','EE','HU','LV','LT','MT','PL',
                   'SK','SI','BG','RO','HR'})

## 2. treated cells to industry grain (single first-treat collapse)

In [3]:
treat = pd.read_parquet(in_treat).rename(columns={'ctry_iso2': 'ctry_iso', 'ind': 'isic3'})

# intensity: one weight per distinct enforcement event (multi-period cases count once)
evt_w = (treat.groupby(['isic3', 'ctry_iso', 'enforcement_event_id'])['w_split']
         .first().reset_index())
w_by_cell = evt_w.groupby(['isic3', 'ctry_iso'])['w_split'].sum().rename('treat_weight')

treat_cell = (treat.groupby(['isic3', 'ctry_iso'])
              .agg(cohort_decision=('cohort_decision', 'min'),   # first-treat
                   cohort_start   =('cohort_start',    'min'),   # earliest entry (staged-in)
                   cohort_end     =('cohort_end',      'max'),   # latest exit  (staged-out)
                   event_ids      =('enforcement_event_id',
                                    lambda s: sorted(set(s.dropna()))))
              .reset_index()
              .merge(w_by_cell, on=['isic3', 'ctry_iso'], how='left'))
treat_cell['n_events']       = treat_cell['event_ids'].apply(len)
treat_cell['repeat_treated'] = (treat_cell['n_events'] > 1).astype(int)

print(f'treated cells: {len(treat_cell)} | industries: {treat_cell["isic3"].nunique()} '
      f'| repeat-treated: {treat_cell["repeat_treated"].sum()}')

treated cells: 1221 | industries: 59 | repeat-treated: 427


## 3. balanced grid + zero-filled outcomes

In [4]:
pat = pd.read_parquet(in_pat).rename(columns={'ind': 'isic3'})
pat = pat[pat['year'] <= year_max].copy()

pat_tot = pat.groupby(['isic3', 'ctry_iso'])['pat_frac'].sum().rename('pat_tot').reset_index()
keep_cells = pd.concat([pat_tot[['isic3', 'ctry_iso']],
                        treat_cell[['isic3', 'ctry_iso']]],
                       ignore_index=True).drop_duplicates()

y_lo, y_hi = int(pat['year'].min()), int(pat['year'].max())
panel = keep_cells.merge(pd.DataFrame({'year': range(y_lo, y_hi + 1)}), how='cross')
panel = panel.merge(pat, on=['isic3', 'ctry_iso', 'year'], how='left')

for col in ['pat_frac', 'n_pat_appln', 'n_applt', 'pat_cit3_frac', 'pat_cit_frac']:
    panel[col] = panel[col].fillna(0)                # true zeros

panel = panel.merge(pat_tot, on=['isic3', 'ctry_iso'], how='left')
panel['pat_tot'] = panel['pat_tot'].fillna(0)

print(f'panel grid: {len(panel):,} cell-years | cells: {len(keep_cells):,} | years {y_lo}-{y_hi}')

panel grid: 413,264 cell-years | cells: 8,984 | years 1978-2023


## 4. merge treatment + event-time (three designs)
`evt_dec` (decision date), `evt_form` (formation), `evt_brk` (breakup) carried as parallel DiD designs.

In [5]:
panel = panel.merge(treat_cell, on=['isic3', 'ctry_iso'], how='left')
panel['ever_treated']   = panel['cohort_decision'].notna().astype(int)
panel['repeat_treated'] = panel['repeat_treated'].fillna(0).astype(int)
panel['n_events']       = panel['n_events'].fillna(0).astype(int)
panel['treat_weight']   = panel['treat_weight'].fillna(0.0)

# per design: event-time, absorbing post, and first_treat (NaN = never-treated sentinel for 04)
for tag, coh in {'dec': 'cohort_decision', 'form': 'cohort_start', 'brk': 'cohort_end'}.items():
    panel[f'evt_{tag}']         = panel['year'] - panel[coh]
    panel[f'treat_{tag}']       = ((panel[coh].notna()) & (panel['year'] >= panel[coh])).astype(int)
    panel[f'first_treat_{tag}'] = panel[coh]

panel['isic2']   = panel['isic3'].str[:2]                        # ISIC division
panel['in_eea']  = panel['ctry_iso'].isin(EEA31).astype(int)     # static EEA-31 membership
dp = (panel['cohort_decision'] >= 2004)
panel['dec_post2004'] = dp.where(panel['cohort_decision'].notna()).astype('Int64')  # 1/0/<NA>

panel['ln1p_pat']  = np.log1p(panel['pat_frac'])
panel['asinh_pat'] = np.arcsinh(panel['pat_frac'])

panel = panel.sort_values(['isic3', 'ctry_iso', 'year']).reset_index(drop=True)
print('columns:', list(panel.columns))

columns: ['isic3', 'ctry_iso', 'year', 'pat_frac', 'pat_cit3_frac', 'pat_cit_frac', 'n_pat_appln', 'n_applt', 'pat_tot', 'cohort_decision', 'cohort_start', 'cohort_end', 'event_ids', 'treat_weight', 'n_events', 'repeat_treated', 'ever_treated', 'evt_dec', 'treat_dec', 'first_treat_dec', 'evt_form', 'treat_form', 'first_treat_form', 'evt_brk', 'treat_brk', 'first_treat_brk', 'isic2', 'in_eea', 'dec_post2004', 'ln1p_pat', 'asinh_pat']


## 5. coverage of treated industries at this grain

In [6]:
tr = panel.loc[panel['ever_treated'] == 1,
               ['isic3', 'ctry_iso', 'pat_tot', 'repeat_treated', 'in_eea']].drop_duplicates()

print(f'treated cells                     : {len(tr)}')
print(f'  clean single-treatment          : {len(tr) - int(tr["repeat_treated"].sum())}')
print(f'  repeat-treated                  : {int(tr["repeat_treated"].sum())}')
print(f'  zero patent activity (pat_tot=0): {int((tr["pat_tot"]==0).sum())}')
print(f'  in-EEA / outside                : {int(tr["in_eea"].sum())} / {int((tr["in_eea"]==0).sum())}')
print(f'treated industries (ISIC3)        : {tr["isic3"].nunique()}')

miss_dec = int(treat_cell['cohort_decision'].isna().sum())
print(f'treated cells missing decision cohort: {miss_dec}')
assert miss_dec == 0, 'treated cell without decision cohort -> ever_treated must be design-specific'

treated cells                     : 1221
  clean single-treatment          : 794
  repeat-treated                  : 427
  zero patent activity (pat_tot=0): 328
  in-EEA / outside                : 1221 / 0
treated industries (ISIC3)        : 59
treated cells missing decision cohort: 0


## 6. diagnostics

In [7]:
n_cells = panel[['isic3', 'ctry_iso']].drop_duplicates().shape[0]
print(f'cells: {n_cells:,} | cell-years: {len(panel):,} | years {panel["year"].min()}-{panel["year"].max()} | countries: {panel["ctry_iso"].nunique()} | industries: {panel["isic3"].nunique()}')

tr_dec = panel.loc[panel['ever_treated'] == 1, ['isic3', 'ctry_iso', 'dec_post2004']].drop_duplicates()
print('\ntreated cells by regime (dec_post2004):')
print(tr_dec['dec_post2004'].value_counts(dropna=False).to_string())

for tag, coh in [('dec', 'cohort_decision'), ('form', 'cohort_start'), ('brk', 'cohort_end')]:
    tc = panel.loc[panel['ever_treated'] == 1, ['isic3', 'ctry_iso', coh]].drop_duplicates()
    cs = tc.groupby(coh).size()
    n_ft = panel.loc[panel['ever_treated'] == 1, ['isic3','ctry_iso',f'first_treat_{tag}']] \
                .drop_duplicates()[f'first_treat_{tag}'].notna().sum()
    print(f'{tag}: {len(cs)} cohorts | singletons {list(cs[cs==1].index)} | first_treat non-null {n_ft}')

print('\n=== outcomes ===')
print(panel[['pat_frac', 'pat_cit3_frac', 'pat_cit_frac']].describe().round(3).to_string())
print('zero pat_frac cell-years:', f'{(panel["pat_frac"]==0).mean():.1%}')

cells: 8,984 | cell-years: 413,264 | years 1978-2023 | countries: 49 | industries: 227

treated cells by regime (dec_post2004):
dec_post2004
1    1019
0     202
dec: 28 cohorts | singletons [np.int64(1996)] | first_treat non-null 1221
form: 29 cohorts | singletons [np.int64(1980), np.int64(1985), np.int64(1991)] | first_treat non-null 1221
brk: 28 cohorts | singletons [np.int64(1990), np.int64(1993), np.int64(1996)] | first_treat non-null 1221

=== outcomes ===
         pat_frac  pat_cit3_frac  pat_cit_frac
count  413264.000     413264.000    413264.000
mean        5.931          2.009         8.467
std        48.043         19.674        75.260
min         0.000          0.000         0.000
25%         0.000          0.000         0.000
50%         0.000          0.000         0.000
75%         0.335          0.011         0.146
max      3413.144       1469.972      5083.501
zero pat_frac cell-years: 52.9%


## 7. save

In [10]:
panel_out = panel.copy()
panel_out['event_ids'] = panel_out['event_ids'].apply(
    lambda x: ','.join(map(str, x)) if isinstance(x, (list, tuple, np.ndarray)) else '')
panel_out.to_parquet(out_panel, index=False)
print(f'saved: {out_panel.name} ({len(panel_out):,} rows, {panel_out.shape[1]} cols)')
print('columns:', list(panel_out.columns))

saved: panel.parquet (413,264 rows, 31 cols)
columns: ['isic3', 'ctry_iso', 'year', 'pat_frac', 'pat_cit3_frac', 'pat_cit_frac', 'n_pat_appln', 'n_applt', 'pat_tot', 'cohort_decision', 'cohort_start', 'cohort_end', 'event_ids', 'treat_weight', 'n_events', 'repeat_treated', 'ever_treated', 'evt_dec', 'treat_dec', 'first_treat_dec', 'evt_form', 'treat_form', 'first_treat_form', 'evt_brk', 'treat_brk', 'first_treat_brk', 'isic2', 'in_eea', 'dec_post2004', 'ln1p_pat', 'asinh_pat']
